In [ ]:
from ingest import load_faq_data

In [ ]:
documents = load_faq_data()

In [ ]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [ ]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

79

In [ ]:
documents = documents_llm

In [ ]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [ ]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
from dotenv import load_dotenv
from groq import Groq
import os
load_dotenv()
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
import json
user_prompt = json.dumps(doc)

In [ ]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [ ]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [ ]:
response = groq_client.chat.completions.create(
    model="gpt-5.4-mini",
    messages = messages,
    text_format=Questions
)

In [ ]:
response.output_parsed.questions

['I just found this course — is it too late to join now?',
 'Can I still enroll if I discovered the course after it started?',
 'If I join late, will I still be able to get a certificate?',
 'What do I need to do to qualify for the certificate if I’m starting now?',
 'Is the only deadline the project submission for getting the certificate?']

In [ ]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [ ]:
from evaluation_utils import llm_structured

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['How is the capstone homework graded, and what do I need to do to get all the points?', 'What are the point values for answering the questions, adding learning resources, and submitting a FAQ question?', 'How many points can I earn from one homework in this project?', 'Does the homework score depend on correct answers only, or are public learning items and FAQ contributions counted too?', 'What tasks should I complete to maximize my homework score and improve my leaderboard position?']


In [ ]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=312)

In [ ]:
from evaluation_utils import calc_price

In [ ]:
calc_price(usage)

{'input_cost': 0.00019725, 'output_cost': 0.0004815, 'total_cost': 0.00067875}

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'How is the capstone homework graded, and what do I need to do to get all the points?',
  'document': '0d200c8c58'},
 {'question': 'What are the point values for answering the questions, adding learning resources, and submitting a FAQ question?',
  'document': '0d200c8c58'},
 {'question': 'How many points can I earn from one homework in this project?',
  'document': '0d200c8c58'},
 {'question': 'Does the homework score depend on correct answers only, or are public learning items and FAQ contributions counted too?',
  'document': '0d200c8c58'},
 {'question': 'What tasks should I complete to maximize my homework score and improve my leaderboard position?',
  'document': '0d200c8c58'}]